# Panzer — Endpoints autenticados (trading)

Este notebook muestra como usar `BinanceClient` para operar con endpoints
que requieren firma HMAC-SHA256.

Las respuestas se muestran tal cual las devuelve la API de Binance — son
exactamente lo que devuelve panzer, sin transformaciones.

**Requisitos:**
- API key y API secret de Binance.
- La primera vez Panzer te las pide por terminal y las almacena cifradas
  en `~/.panzer_creds`. Las siguientes veces las carga automaticamente.

> **ATENCION:** Los ejemplos de ordenes estan comentados para evitar
> operaciones accidentales con dinero real.

## 1. Crear el cliente

In [ ]:
from panzer import BinanceClient

client = BinanceClient(market="spot")

## 2. account() — respuesta cruda

La respuesta de `/api/v3/account` es un dict grande. Mostramos las claves
de primer nivel y luego los balances con saldo.

In [ ]:
account = client.account()

# Claves que trae la respuesta
print("Claves de la respuesta account():")
print(list(account.keys()))

In [ ]:
# Campos escalares de la respuesta (todo menos balances y permissions)
{k: v for k, v in account.items() if k not in ("balances", "permissions")}

In [ ]:
# Balances con saldo > 0 — cada elemento tal cual viene de Binance
[b for b in account["balances"] if float(b["free"]) > 0 or float(b["locked"]) > 0]

## 3. my_trades() — respuesta cruda

Cada trade es un dict con la estructura nativa de Binance.

In [ ]:
my_trades = client.my_trades("BTCUSDT", limit=3)
my_trades

## 4. open_orders() — respuesta cruda

In [ ]:
# Lista vacia si no hay ordenes abiertas
open_orders = client.open_orders("BTCUSDT")
open_orders

## 5. all_orders() — respuesta cruda

Historial completo: abiertas, cerradas, canceladas. Cada orden es un dict
con todos los campos que devuelve Binance.

In [ ]:
all_orders = client.all_orders("BTCUSDT", limit=3)
all_orders

## 6. Crear ordenes

> **CUIDADO:** Estas celdas crean ordenes REALES. Estan comentadas por seguridad.

### Orden LIMIT — respuesta esperada

```python
{
    "symbol": "BTCUSDT",
    "orderId": 28,
    "orderListId": -1,
    "clientOrderId": "6gCrw2kRUAF9CvJDGP16IP",
    "transactTime": 1507725176595,
    "price": "50000.00000000",
    "origQty": "0.00100000",
    "executedQty": "0.00000000",
    "cummulativeQuoteQty": "0.00000000",
    "status": "NEW",
    "timeInForce": "GTC",
    "type": "LIMIT",
    "side": "BUY",
    "workingTime": 1507725176595,
    "fills": [],
    "selfTradePreventionMode": "EXPIRE_TAKER"
}
```

In [ ]:
# DESCOMENTA PARA EJECUTAR:

# order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="LIMIT",
#     quantity=0.001,
#     price=50000.0,
#     time_in_force="GTC",
# )
# order  # respuesta cruda de Binance

### Orden MARKET — respuesta esperada

Las ordenes MARKET se ejecutan inmediatamente. La respuesta incluye `fills`
con los precios y cantidades reales de ejecucion.

```python
{
    "symbol": "BTCUSDT",
    "orderId": 29,
    "status": "FILLED",
    "type": "MARKET",
    "side": "BUY",
    "executedQty": "0.00010000",
    "cummulativeQuoteQty": "9.12340000",
    "fills": [
        {
            "price": "91234.00000000",
            "qty": "0.00010000",
            "commission": "0.00000010",
            "commissionAsset": "BTC",
            "tradeId": 123456
        }
    ]
}
```

In [ ]:
# DESCOMENTA PARA EJECUTAR:

# market_order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="MARKET",
#     quote_order_qty=10.0,   # comprar 10 USDT de BTC
# )
# market_order  # respuesta cruda

### Cancelar una orden

In [ ]:
# DESCOMENTA PARA EJECUTAR:

# cancelled = client.cancel_order("BTCUSDT", order_id=12345678)
# cancelled  # respuesta cruda

## 7. signed_request() — endpoint generico

Para cualquier endpoint autenticado sin wrapper dedicado, usa
`signed_request()`. Panzer anade timestamp, firma y API key automaticamente.

In [ ]:
# Ejemplo: obtener depositAddress (requiere firma completa)
# data = client.signed_request(
#     "GET",
#     "/sapi/v1/capital/deposit/address",
#     params=[("coin", "BTC"), ("network", "BTC")],
# )
# data  # respuesta cruda

# Ejemplo: listen key (semi-firmado, solo API key, sin HMAC)
# listen = client.signed_request(
#     "POST",
#     "/api/v3/userDataStream",
#     sign=False,
# )
# listen  # {"listenKey": "pqia91ma19a5s61cv6a81va65sdf19v8a65a1a5s61cv6a81va65sdf19v8a65a1"}

## 8. Futuros (UM / CM)

Los mismos metodos funcionan con futuros. Solo cambia el `market` al crear
el cliente. Panzer resuelve automaticamente los endpoints correctos.

```python
# USDT-M Futures
um_client = BinanceClient(market="um")
um_account = um_client.account()           # /fapi/v2/account
um_trades = um_client.my_trades("BTCUSDT") # /fapi/v1/userTrades
um_orders = um_client.all_orders("BTCUSDT")# /fapi/v1/allOrders

# COIN-M Futures
cm_client = BinanceClient(market="cm")
cm_account = cm_client.account()                # /dapi/v1/account
cm_trades = cm_client.my_trades("BTCUSD_PERP")  # /dapi/v1/userTrades
```

Los endpoints privados por mercado:

| Metodo | spot | um | cm |
|---|---|---|---|
| `account()` | /api/v3/account | /fapi/v2/account | /dapi/v1/account |
| `my_trades()` | /api/v3/myTrades | /fapi/v1/userTrades | /dapi/v1/userTrades |
| `new_order()` | /api/v3/order | /fapi/v1/order | /dapi/v1/order |
| `open_orders()` | /api/v3/openOrders | /fapi/v1/openOrders | /dapi/v1/openOrders |
| `all_orders()` | /api/v3/allOrders | /fapi/v1/allOrders | /dapi/v1/allOrders |

## 9. Manejo de errores

Panzer levanta `BinanceAPIException` con el codigo y mensaje originales
de Binance. No transforma ni oculta nada.

```python
from panzer.errors import BinanceAPIException

try:
    client.my_trades("SIMBOLO_INVALIDO")
except BinanceAPIException as e:
    print(e.status_code)       # 400
    print(e.error_payload.code) # -1121
    print(e.error_payload.msg)  # "Invalid symbol."
    print(e.body)              # JSON crudo de la respuesta
```

Codigos de error comunes de Binance:

| Codigo | Significado |
|---|---|
| -1121 | Invalid symbol |
| -1021 | Timestamp fuera de recvWindow |
| -2010 | Insufficient balance |
| -2013 | Order does not exist |
| -1003 | Too many requests (rate limit) |